In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import sklearn
import nflreadpy as nfl

# NFL Win Prediction - Data Exploration

This notebook explores the historical NFL dataset that will eventually be used to train a binary classification model.

The initial goals are to:

1. Load the required Python libraries.
2. Load historical NFL game data.
3. Understand the structure of the dataset.
4. Identify useful columns.
5. Identify potential sources of data leakage.

In [2]:
print("pands version:", pd.__version__)
print("numpy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)

pands version: 3.0.6
numpy version: 2.5.3
scikit-learn version: 1.9.1


In [3]:
import sys 

print(sys.executable)
print(sys.version)


c:\Users\owens\anaconda3\envs\nfl-ml\python.exe
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]


## Load Historical NFL Schedule Data

We will begin by loading historical NFL schedules and game results.

The goal at this stage is not to build a model yet. It is to understand the structure of the dataset and determine what information is available for each game.

In [4]:
seasons = list(range(2015, 2026))

schedules = nfl.load_schedules(seasons)

In [5]:
games = schedules.to_pandas()

In [6]:
games.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field


In [7]:
games.shape

(3028, 46)

In [8]:
games.columns.tolist()

['game_id',
 'season',
 'game_type',
 'week',
 'gameday',
 'weekday',
 'gametime',
 'away_team',
 'away_score',
 'home_team',
 'home_score',
 'location',
 'result',
 'total',
 'overtime',
 'old_game_id',
 'gsis',
 'nfl_detail_id',
 'pfr',
 'pff',
 'espn',
 'ftn',
 'away_rest',
 'home_rest',
 'away_moneyline',
 'home_moneyline',
 'spread_line',
 'away_spread_odds',
 'home_spread_odds',
 'total_line',
 'under_odds',
 'over_odds',
 'div_game',
 'roof',
 'surface',
 'temp',
 'wind',
 'away_qb_id',
 'home_qb_id',
 'away_qb_name',
 'home_qb_name',
 'away_coach',
 'home_coach',
 'referee',
 'stadium_id',
 'stadium']

In [9]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3028 entries, 0 to 3027
Data columns (total 46 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   game_id           3028 non-null   str    
 1   season            3028 non-null   int32  
 2   game_type         3028 non-null   str    
 3   week              3028 non-null   int32  
 4   gameday           3028 non-null   str    
 5   weekday           3028 non-null   str    
 6   gametime          3028 non-null   str    
 7   away_team         3028 non-null   str    
 8   away_score        3028 non-null   int32  
 9   home_team         3028 non-null   str    
 10  home_score        3028 non-null   int32  
 11  location          3028 non-null   str    
 12  result            3028 non-null   int32  
 13  total             3028 non-null   int32  
 14  overtime          3028 non-null   int32  
 15  old_game_id       3028 non-null   str    
 16  gsis              3028 non-null   int32  
 17  nfl_de

In [10]:
games["season"].value_counts().sort_index

<bound method Series.sort_index of season
2021    285
2023    285
2024    285
2025    285
2022    284
2020    269
2015    267
2016    267
2017    267
2018    267
2019    267
Name: count, dtype: int64>

In [11]:
games.sample(5)

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
2243,2023_05_NO_NE,2023,REG,5,2023-10-08,Sunday,13:00,NO,34,NE,...,10.0,00-0031280,00-0036972,Derek Carr,Mac Jones,Dennis Allen,Bill Belichick,Alex Kemp,BOS00,Gillette Stadium
2541,2024_06_TB_NO,2024,REG,6,2024-10-13,Sunday,13:00,TB,51,NO,...,NaN,00-0034855,00-0039376,Baker Mayfield,Spencer Rattler,Todd Bowles,Dennis Allen,Clete Blakeman,NOR00,Mercedes-Benz Superdome
2579,2024_08_DAL_SF,2024,REG,8,2024-10-27,Sunday,20:20,DAL,24,SF,...,9.0,00-0033077,00-0037834,Dak Prescott,Brock Purdy,Mike McCarthy,Kyle Shanahan,Bill Vinovich,SFO01,Levi's Stadium
2327,2023_11_LAC_GB,2023,REG,11,2023-11-19,Sunday,13:00,LAC,20,GB,...,5.0,00-0036355,00-0036264,Justin Herbert,Jordan Love,Brandon Staley,Matt LaFleur,John Hussey,GNB00,Lambeau Field
1522,2020_13_NE_LAC,2020,REG,13,2020-12-06,Sunday,16:25,NE,45,LAC,...,NaN,00-0027939,00-0036355,Cam Newton,Justin Herbert,Bill Belichick,Anthony Lynn,Carl Cheffers,LAX01,SoFi Stadium


## Build the Game Backbone

The schedule dataset will serve as the backbone of the project.

Each row represents one NFL game. Additional team, player, matchup, and contextual features will later be attached to these rows using identifiers such as season, week, team, and game ID.

Before building features, the schedule data must be cleaned and the prediction target must be defined.

In [12]:
games["game_type"].value_counts()

game_type
REG    2895
WC       56
DIV      44
CON      22
SB       11
Name: count, dtype: int64

In [13]:
regular_games = games.loc[games["game_type"] == "REG"].copy()

In [14]:
TEAM_ABBR_MAP = {
    "OAK": "LV",
    "SD": "LAC",
    "STL": "LA"
}

regular_games["home_team"] = (
    regular_games["home_team"]
    .replace(TEAM_ABBR_MAP)
)

regular_games["away_team"] = (
    regular_games["away_team"]
    .replace(TEAM_ABBR_MAP)
)

In [15]:
games.loc[...]

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3023,2025_20_LA_CHI,2025,DIV,20,2026-01-18,Sunday,18:30,LA,20,CHI,...,6.0,00-0026498,00-0039918,Matthew Stafford,Caleb Williams,Sean McVay,Ben Johnson,Shawn Hochuli,CHI98,Soldier Field
3024,2025_20_HOU_NE,2025,DIV,20,2026-01-18,Sunday,15:00,HOU,16,NE,...,2.0,00-0039163,00-0039851,C.J. Stroud,Drake Maye,DeMeco Ryans,Mike Vrabel,Shawn Smith,BOS00,Gillette Stadium
3025,2025_21_NE_DEN,2025,CON,21,2026-01-25,Sunday,15:00,NE,10,DEN,...,10.0,00-0039851,00-0035264,Drake Maye,Jarrett Stidham,Mike Vrabel,Sean Payton,Alex Kemp,DEN00,Empower Field at Mile High
3026,2025_21_LA_SEA,2025,CON,21,2026-01-25,Sunday,18:30,LA,27,SEA,...,2.0,00-0026498,00-0034869,Matthew Stafford,Sam Darnold,Sean McVay,Mike Macdonald,Clay Martin,SEA00,Lumen Field


In [16]:
regular_games.shape

(2895, 46)

In [17]:
backbone_columns = [
    "game_id",
    "season",
    "week",
    "gameday",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "home_rest",
    "away_rest",
    "location"
]

regular_games[backbone_columns].head(10)

,game_id,season,week,gameday,home_team,away_team,home_score,away_score,home_rest,away_rest,location
0,2015_01_PIT_NE,2015,1,2015-09-10,NE,PIT,28,21,7,7,Home
1,2015_01_IND_BUF,2015,1,2015-09-13,BUF,IND,27,14,7,7,Home
2,2015_01_GB_CHI,2015,1,2015-09-13,CHI,GB,23,31,7,7,Home
3,2015_01_KC_HOU,2015,1,2015-09-13,HOU,KC,20,27,7,7,Home
4,2015_01_CAR_JAX,2015,1,2015-09-13,JAX,CAR,9,20,7,7,Home
5,2015_01_CLE_NYJ,2015,1,2015-09-13,NYJ,CLE,31,10,7,7,Home
6,2015_01_SEA_STL,2015,1,2015-09-13,LA,SEA,34,31,7,7,Home
7,2015_01_MIA_WAS,2015,1,2015-09-13,WAS,MIA,10,17,7,7,Home
8,2015_01_NO_ARI,2015,1,2015-09-13,ARI,NO,31,19,7,7,Home
9,2015_01_DET_SD,2015,1,2015-09-13,LAC,DET,33,28,7,7,Home


In [18]:
regular_games[["home_score", "away_score"]].isna().sum()

home_score    0
away_score    0
dtype: int64

In [19]:
(regular_games["home_score"] == regular_games["away_score"]).sum()

np.int64(10)

In [20]:
game_backbone = regular_games.loc[
    regular_games["home_score"] != regular_games["away_score"],
    backbone_columns
    ].copy()

In [21]:
game_backbone.head()

,game_id,season,week,gameday,home_team,away_team,home_score,away_score,home_rest,away_rest,location
0,2015_01_PIT_NE,2015,1,2015-09-10,NE,PIT,28,21,7,7,Home
1,2015_01_IND_BUF,2015,1,2015-09-13,BUF,IND,27,14,7,7,Home
2,2015_01_GB_CHI,2015,1,2015-09-13,CHI,GB,23,31,7,7,Home
3,2015_01_KC_HOU,2015,1,2015-09-13,HOU,KC,20,27,7,7,Home
4,2015_01_CAR_JAX,2015,1,2015-09-13,JAX,CAR,9,20,7,7,Home


In [22]:
game_backbone["home_win"] = (
    game_backbone["home_score"] > game_backbone["away_score"]
).astype(int)

In [23]:
game_backbone[["season", "week", "away_team", "home_team", "away_score", "home_score", "home_win"]].sample(10)

,season,week,away_team,home_team,away_score,home_score,home_win
141,2015,10,MIN,LV,30,14,0
2018,2022,9,IND,NE,3,26,1
821,2018,2,CLE,NO,18,21,1
1618,2021,1,CHI,LA,14,34,1
843,2018,3,GB,WAS,17,31,1
537,2017,1,BAL,CIN,20,0,0
98,2015,7,CLE,LA,6,24,1
975,2018,12,GB,MIN,17,24,1
1158,2019,6,PIT,LAC,24,17,0
2471,2024,1,WAS,TB,20,37,1


In [24]:
game_backbone["home_win"].value_counts()

home_win
1    1575
0    1310
Name: count, dtype: int64

In [25]:
game_backbone["home_win"].value_counts(normalize=True)

home_win
1    0.545927
0    0.454073
Name: proportion, dtype: float64

In [26]:
games.to_csv(
    "../data/raw/schedules_2015_2025.csv",
    index=False
)

game_backbone.to_csv(
    "../data/processed/game_backbone.csv",
    index=False
)

## Creating Pregame Team Features

Machine-learning features must represent information that was available before each game.

To calculate team form, the game-level schedule data will first be transformed into a team-level dataset where each game appears twice: once from each team's perspective.

Initial features will include:

- Recent win percentage
- Recent points scored
- Recent points allowed
- Recent point differential

All rolling statistics will use only games played before the game being predicted.

In [27]:
feature_games = regular_games.dropna(
    subset=["home_score", "away_score"]
).copy()

feature_games["gameday"] = pd.to_datetime(
    feature_games["gameday"]
)

In [28]:
feature_games["home_result_value"] = np.select(
    [
        feature_games["home_score"] > feature_games["away_score"],
        feature_games["home_score"] == feature_games["away_score"]
    ],
    [
        1.0,
        0.5
    ],
    default=0.0
)

feature_games["away_result_value"] = (
    1-feature_games["home_result_value"]
)

In [29]:
home_rows = feature_games[
    [
        "game_id",
        "season",
        "week",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_result_value"
    ]
].rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_for",
        "away_score": "points_against",
        "home_result_value": "result_value"
    }
)

home_rows["is_home"] = 1

In [30]:
away_rows = feature_games[
    [
        "game_id",
        "season",
        "week",
        "gameday",
        "away_team",
        "home_team",
        "away_score",
        "home_score",
        "away_result_value"
    ]
].rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_for",
        "home_score": "points_against",
        "away_result_value": "result_value"
    }
)

away_rows["is_home"] = 0

In [31]:
team_games = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

In [32]:
team_games["point_diff"] = (
    team_games["points_for"]
    - team_games["points_against"]
)

In [33]:
team_games = team_games.sort_values(
    ["season", "team", "gameday"]
).reset_index(drop=True)

In [34]:
team_games.shape

(5790, 11)

In [35]:
team_games.head(10)

,game_id,season,week,gameday,team,opponent,points_for,points_against,result_value,is_home,point_diff
0,2015_01_NO_ARI,2015,1,2015-09-13,ARI,NO,31,19,1.0,1,12
1,2015_02_ARI_CHI,2015,2,2015-09-20,ARI,CHI,48,23,1.0,0,25
2,2015_03_SF_ARI,2015,3,2015-09-27,ARI,SF,47,7,1.0,1,40
3,2015_04_STL_ARI,2015,4,2015-10-04,ARI,LA,22,24,0.0,1,-2
4,2015_05_ARI_DET,2015,5,2015-10-11,ARI,DET,42,17,1.0,0,25
5,2015_06_ARI_PIT,2015,6,2015-10-18,ARI,PIT,13,25,0.0,0,-12
6,2015_07_BAL_ARI,2015,7,2015-10-26,ARI,BAL,26,18,1.0,1,8
7,2015_08_ARI_CLE,2015,8,2015-11-01,ARI,CLE,34,20,1.0,0,14
8,2015_10_ARI_SEA,2015,10,2015-11-15,ARI,SEA,39,32,1.0,0,7
9,2015_11_CIN_ARI,2015,11,2015-11-22,ARI,CIN,34,31,1.0,1,3


In [36]:
team_group = team_games.groupby(
    ["season", "team"]
)

In [37]:
team_games["win_pct_last5"] = (
    team_group["result_value"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

In [38]:
team_games["points_for_last5"] = (
    team_group["points_for"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

team_games["points_against_last5"] = (
    team_group["points_against"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

team_games["point_diff_last5"] = (
    team_group["point_diff"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

In [39]:
team_games.loc[
    (team_games["team"] == "CIN") &
    (team_games["season"] == 2025),
    [
        "week",
        "opponent",
        "points_for",
        "points_against",
        "result_value",
        "win_pct_last5",
        "points_for_last5",
        "points_against_last5",
        "point_diff_last5"
    ]
].head(8)

,week,opponent,points_for,points_against,result_value,win_pct_last5,points_for_last5,points_against_last5,point_diff_last5
5348,1,CLE,17,16,1.0,NaN,NaN,NaN,NaN
5349,2,JAX,31,27,1.0,1.000000,17.000000,16.000000,1.0
5350,3,MIN,10,48,0.0,1.000000,24.000000,21.500000,2.5
5351,4,DEN,3,28,0.0,0.666667,19.333333,30.333333,-11.0
5352,5,DET,24,37,0.0,0.500000,15.250000,29.750000,-14.5
5353,6,GB,18,27,0.0,0.400000,17.000000,31.200000,-14.2
5354,7,PIT,33,31,1.0,0.200000,17.200000,33.400000,-16.2
5355,8,NYJ,38,39,0.0,0.200000,17.600000,34.200000,-16.6


## Loading Weekly Team Performance Data

The schedule dataset provides the game backbone, but it does not contain enough football-performance information for a meaningful prediction model.

Weekly team statistics will now be loaded so that offensive performance can be measured before each game.

These statistics will later be converted into rolling pregame features, just like the recent-form features above.

In [40]:
weekly_stats = nfl.load_team_stats(
    seasons=seasons,
    summary_level="week"
).to_pandas()

In [41]:
weekly_stats.shape

(6056, 138)

In [42]:
weekly_stats.columns.tolist()

['season',
 'week',
 'team',
 'season_type',
 'game_id',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'receiving_10',
 'receiving_16',
 'receiving_20',
 'receiving_40',
 'special_teams_tds',
 'def_ta

In [43]:
weekly_stats.head()

,season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,...,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards
0,2015,1,ARI,REG,2015_01_NO_ARI,NO,19,32,307,3,...,158,2,1,1,0,1,1,2,0,156
1,2015,1,ATL,REG,2015_01_PHI_ATL,PHI,23,34,298,2,...,231,3,1,0,0,0,3,25,0,206
2,2015,1,BAL,REG,2015_01_BAL_DEN,DEN,18,32,117,0,...,310,2,2,0,0,1,3,20,0,290
3,2015,1,BUF,REG,2015_01_IND_BUF,IND,14,19,195,1,...,252,2,0,0,1,2,2,1,0,231
4,2015,1,CAR,REG,2015_01_CAR_JAX,JAX,18,31,175,1,...,179,2,0,0,1,2,1,9,0,150


In [44]:
weekly_stats["season_type"].value_counts()

season_type
REG     5790
POST     266
Name: count, dtype: int64

In [45]:
weekly_reg = weekly_stats.loc[
    weekly_stats["season_type"] == "REG"
].copy()

In [46]:
weekly_reg["team"] = (
    weekly_reg["team"]
    .replace(TEAM_ABBR_MAP)
)

weekly_reg["opponent_team"] = (
    weekly_reg["opponent_team"]
    .replace(TEAM_ABBR_MAP)
)

In [47]:
weekly_reg.shape

(5790, 138)

## Creating Team Efficiency Metrics

Raw totals such as passing yards and rushing yards can be affected by game volume and game script.

To better measure team quality, several efficiency statistics will be derived from the weekly data.

Initial efficiency metrics include:

- Passing EPA per dropback
- Rushing EPA per carry
- Sack rate
- Turnovers

In [48]:
weekly_reg["dropbacks"] = (
    weekly_reg["attempts"]
    + weekly_reg["sacks_suffered"]
)

In [49]:
weekly_reg["pass_epa_per_dropback"] = (
    weekly_reg["passing_epa"]
    / weekly_reg["dropbacks"]
)

weekly_reg["rush_epa_per_carry"] = (
    weekly_reg["rushing_epa"]
    / weekly_reg["carries"]
)

weekly_reg["sack_rate"] = (
    weekly_reg["sacks_suffered"]
    / weekly_reg["dropbacks"]
)

weekly_reg["turnovers"] = (
    weekly_reg["passing_interceptions"] 
    + weekly_reg["fumbles_lost_total"]
)

In [50]:
opponent_stats = weekly_reg[
    [
        "game_id",
        "team",
        "opponent_team",
        "passing_epa",
        "dropbacks",
        "rushing_epa",
        "carries",
        "turnovers"
    ]
].copy()

opponent_stats = opponent_stats.rename(
    columns={
        "team": "opponent_team",
        "opponent_team": "team",
        "passing_epa": "passing_epa_allowed",
        "dropbacks": "def_dropbacks",
        "rushing_epa": "rushing_epa_allowed",
        "carries": "def_carries",
        "turnovers": "takeaways"
    }
)

In [51]:
team_performance = weekly_reg.merge(
    opponent_stats,
    on=["game_id", "team", "opponent_team"],
    how="left",
    validate="one_to_one"
)

team_performance = team_performance.sort_values(
    ["season", "team", "week"]
).reset_index(drop=True)

In [52]:
team_performance["pass_epa_allowed_per_dropback"] = (
    team_performance["passing_epa_allowed"]
    / team_performance["def_dropbacks"]
)

team_performance["rush_epa_allowed_per_carry"] = (
    team_performance["rushing_epa_allowed"]
    / team_performance["def_carries"]
)

team_performance["def_sack_rate"] = (
    team_performance["def_sacks"]
    / team_performance["def_dropbacks"]
)

In [53]:
required_columns = [
    "passing_epa",
    "dropbacks",
    "pass_epa_per_dropback",
    "rushing_epa",
    "carries",
    "rush_epa_per_carry",
    "sacks_suffered",
    "sack_rate",
    "turnovers",
    "passing_epa_allowed",
    "def_dropbacks",
    "pass_epa_allowed_per_dropback",
    "rushing_epa_allowed",
    "def_carries",
    "rush_epa_allowed_per_carry",
    "def_sacks",
    "def_sack_rate",
    "takeaways"
]

missing_columns = [
    col for col in required_columns
    if col not in team_performance.columns
]

print("Missing columns:", missing_columns)
print("Shape:", team_performance.shape)

Missing columns: []
Shape: (5790, 151)


In [54]:
team_performance[
    [
        "season",
        "week",
        "team",
        "opponent_team",

        "pass_epa_per_dropback",
        "pass_epa_allowed_per_dropback",

        "rush_epa_per_carry",
        "rush_epa_allowed_per_carry",

        "sack_rate",
        "def_sack_rate",

        "turnovers",
        "takeaways"
    ]
].head(10)

,season,week,team,opponent_team,pass_epa_per_dropback,pass_epa_allowed_per_dropback,rush_epa_per_carry,rush_epa_allowed_per_carry,sack_rate,def_sack_rate,turnovers,takeaways
0,2015,1,ARI,NO,0.507832,0.036596,-0.348388,-0.187498,0.000000,0.040000,1,1
1,2015,2,ARI,CHI,0.469720,-0.267022,0.060615,-0.152335,0.000000,0.058824,2,2
2,2015,3,ARI,SF,0.341914,-1.139059,0.134756,-0.116111,0.030303,0.095238,1,4
3,2015,4,ARI,LA,0.092647,0.172544,0.007977,-0.048191,0.080000,0.040000,3,0
4,2015,5,ARI,DET,0.331412,-0.152221,0.330461,-0.402883,0.100000,0.014085,0,6
5,2015,6,ARI,PIT,0.153256,0.201343,-0.251504,-0.116675,0.021739,0.047619,3,0
6,2015,7,ARI,BAL,0.313432,0.135927,0.006264,-0.213989,0.064516,0.069767,0,2
7,2015,8,ARI,CLE,0.652758,0.027427,-0.375199,-0.481185,0.025641,0.024390,4,2
8,2015,10,ARI,SEA,0.006519,-0.220813,-0.126162,0.328179,0.060000,0.029412,3,1
9,2015,11,ARI,CIN,0.361262,0.225961,-0.116642,-0.152791,0.060606,0.093023,2,1


In [55]:
team_group = team_performance.groupby(
    ["season", "team"]
)

team_performance["passing_epa_last5_total"] = (
    team_group["passing_epa"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["dropbacks_last5_total"] = (
    team_group["dropbacks"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["rushing_epa_last5_total"] = (
    team_group["rushing_epa"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["carries_last5_total"] = (
    team_group["carries"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["sacks_suffered_last5_total"] = (
    team_group["sacks_suffered"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["pass_epa_per_dropback_last5"] = (
    team_performance["passing_epa_last5_total"]
    / team_performance["dropbacks_last5_total"]
)

team_performance["rush_epa_per_carry_last5"] = (
    team_performance["rushing_epa_last5_total"]
    / team_performance["carries_last5_total"]
)

team_performance["sack_rate_last5"] = (
    team_performance["sacks_suffered_last5_total"]
    / team_performance["dropbacks_last5_total"]
)

In [56]:
team_performance["passing_epa_allowed_last5_total"] = (
    team_group["passing_epa_allowed"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["def_dropbacks_last5_total"] = (
    team_group["def_dropbacks"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["rushing_epa_allowed_last5_total"] = (
    team_group["rushing_epa_allowed"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["def_carries_last5_total"] = (
    team_group["def_carries"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["def_sacks_last5_total"] = (
    team_group["def_sacks"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )
)

team_performance["pass_epa_allowed_per_dropback_last5"] = (
    team_performance["passing_epa_allowed_last5_total"]
    / team_performance["def_dropbacks_last5_total"]
)

team_performance["rush_epa_allowed_per_carry_last5"] = (
    team_performance["rushing_epa_allowed_last5_total"]
    / team_performance["def_carries_last5_total"]
)

team_performance["def_sack_rate_last5"] = (
    team_performance["def_sacks_last5_total"]
    / team_performance["def_dropbacks_last5_total"]
)

In [57]:
team_performance["turnovers_last5"] = (
    team_group["turnovers"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

team_performance["takeaways_last5"] = (
    team_group["takeaways"]
    .transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
)

In [58]:
team_performance.loc[
    (team_performance["team"] == "CIN") &
    (team_performance["season"] == 2025),
    [
        "week",
        "opponent_team",
        "pass_epa_per_dropback_last5",
        "pass_epa_allowed_per_dropback_last5",
        "rush_epa_per_carry_last5",
        "rush_epa_allowed_per_carry_last5",
        "sack_rate_last5",
        "def_sack_rate_last5",
        "turnovers_last5",
        "takeaways_last5"
    ]
].head(8)

,week,opponent_team,pass_epa_per_dropback_last5,pass_epa_allowed_per_dropback_last5,rush_epa_per_carry_last5,rush_epa_allowed_per_carry_last5,sack_rate_last5,def_sack_rate_last5,turnovers_last5,takeaways_last5
5348,1,CLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5349,2,JAX,-0.012754,-0.039194,-0.217519,-0.074718,0.115385,0.042553,0.000000,2.000000
5350,3,MIN,-0.029187,0.035148,-0.091013,-0.005746,0.081081,0.033333,1.500000,2.000000
5351,4,DEN,-0.204470,0.075076,-0.176628,0.013510,0.095238,0.051282,2.666667,1.333333
5352,5,DET,-0.239278,0.138766,-0.147197,0.042139,0.097744,0.037736,2.000000,1.250000
5353,6,GB,-0.231944,0.187576,-0.102010,0.008360,0.085714,0.053476,2.200000,1.200000
5354,7,PIT,-0.173139,0.265119,-0.061417,0.049321,0.066667,0.053892,2.200000,1.000000
5355,8,NYJ,-0.085663,0.332793,-0.012316,0.083108,0.061224,0.050633,1.600000,1.000000


## Building the Game-Level Training Dataset

The team-level rolling statistics will now be merged back onto the game-level backbone.

Each game will contain:

- Home team pregame performance
- Away team pregame performance
- Matchup-specific differences
- Game context
- Target variable: `home_win`

Only statistics calculated from games played before kickoff will be used as model features.

In [59]:
team_feature_columns = [
    "game_id",
    "team",

    "pass_epa_per_dropback_last5",
    "rush_epa_per_carry_last5",
    "sack_rate_last5",
    "turnovers_last5",

    "pass_epa_allowed_per_dropback_last5",
    "rush_epa_allowed_per_carry_last5",
    "def_sack_rate_last5",
    "takeaways_last5"
]

In [60]:
home_features = team_performance[
    team_feature_columns
].copy()

In [61]:
home_features = home_features.rename(
    columns={
        "team": "home_team",

        "pass_epa_per_dropback_last5":
            "home_pass_epa_per_dropback_last5",

        "rush_epa_per_carry_last5":
            "home_rush_epa_per_carry_last5",

        "sack_rate_last5":
            "home_sack_rate_last5",

        "turnovers_last5":
            "home_turnovers_last5",

        "pass_epa_allowed_per_dropback_last5":
            "home_pass_epa_allowed_last5",

        "rush_epa_allowed_per_carry_last5":
            "home_rush_epa_allowed_last5",

        "def_sack_rate_last5":
            "home_def_sack_rate_last5",

        "takeaways_last5":
            "home_takeaways_last5"
    }
)

In [62]:
away_features = team_performance[
    team_feature_columns
].copy()

In [63]:
away_features = away_features.rename(
    columns={
        "team": "away_team",

        "pass_epa_per_dropback_last5":
            "away_pass_epa_per_dropback_last5",

        "rush_epa_per_carry_last5":
            "away_rush_epa_per_carry_last5",

        "sack_rate_last5":
            "away_sack_rate_last5",

        "turnovers_last5":
            "away_turnovers_last5",

        "pass_epa_allowed_per_dropback_last5":
            "away_pass_epa_allowed_last5",

        "rush_epa_allowed_per_carry_last5":
            "away_rush_epa_allowed_last5",

        "def_sack_rate_last5":
            "away_def_sack_rate_last5",

        "takeaways_last5":
            "away_takeaways_last5"
    }
)

In [64]:
model_data = game_backbone.copy()

In [65]:
model_data = model_data.merge(
    home_features,
    on=["game_id", "home_team"],
    how="left",
    validate="one_to_one"
)

model_data = model_data.merge(
    away_features,
    on=["game_id", "away_team"],
    how="left",
    validate="one_to_one"
)

In [66]:
model_data.shape

(2885, 28)

In [67]:
print("game backbone rows:", len(game_backbone))
print("model dataset rows:" , len(model_data))

game backbone rows: 2885
model dataset rows: 2885


In [68]:
model_data[
    [
        "season",
        "week",
        "home_team",
        "away_team",
        "home_win",

        "home_pass_epa_per_dropback_last5",
        "away_pass_epa_per_dropback_last5"
    ]
].head(20)

,season,week,home_team,away_team,home_win,home_pass_epa_per_dropback_last5,away_pass_epa_per_dropback_last5
0,2015,1,NE,PIT,1,NaN,NaN
1,2015,1,BUF,IND,1,NaN,NaN
2,2015,1,CHI,GB,0,NaN,NaN
3,2015,1,HOU,KC,0,NaN,NaN
4,2015,1,JAX,CAR,0,NaN,NaN
5,2015,1,NYJ,CLE,1,NaN,NaN
6,2015,1,LA,SEA,1,NaN,NaN
7,2015,1,WAS,MIA,0,NaN,NaN
8,2015,1,ARI,NO,1,NaN,NaN
9,2015,1,LAC,DET,1,NaN,NaN


In [69]:
required_model_columns = [
    "season",
    "week",
    "home_team",
    "away_team",
    "home_win",
    "home_rest",
    "away_rest",

    "home_pass_epa_per_dropback_last5",
    "away_pass_epa_per_dropback_last5",

    "home_rush_epa_per_carry_last5",
    "away_rush_epa_per_carry_last5",

    "home_sack_rate_last5",
    "away_sack_rate_last5",

    "home_turnovers_last5",
    "away_turnovers_last5",

    "home_pass_epa_allowed_last5",
    "away_pass_epa_allowed_last5",

    "home_rush_epa_allowed_last5",
    "away_rush_epa_allowed_last5",

    "home_def_sack_rate_last5",
    "away_def_sack_rate_last5",

    "home_takeaways_last5",
    "away_takeaways_last5"
]

missing_columns = [
    col for col in required_model_columns
    if col not in model_data.columns
]

print("Missing columns:", missing_columns)

assert not missing_columns, (
    f"Missing required model columns: {missing_columns}"
)

Missing columns: []


In [70]:
model_data["rest_diff"] = (
    model_data["home_rest"]
    - model_data["away_rest"]
)

In [71]:
profile_features = [
    "home_pass_epa_per_dropback_last5",
    "away_pass_epa_per_dropback_last5",

    "home_rush_epa_per_carry_last5",
    "away_rush_epa_per_carry_last5",

    "home_sack_rate_last5",
    "away_sack_rate_last5",

    "home_turnovers_last5",
    "away_turnovers_last5",

    "home_pass_epa_allowed_last5",
    "away_pass_epa_allowed_last5",

    "home_rush_epa_allowed_last5",
    "away_rush_epa_allowed_last5",

    "home_def_sack_rate_last5",
    "away_def_sack_rate_last5",

    "home_takeaways_last5",
    "away_takeaways_last5",

    "rest_diff"
]

model_data[profile_features].isna().sum()

model_data_clean = model_data.dropna(
    subset=profile_features
).copy()

In [72]:
print("before:", model_data.shape)
print("after:", model_data_clean.shape)

before: (2885, 29)
after: (2711, 29)


In [84]:
schedule_teams = set(
    game_backbone["home_team"]
).union(game_backbone["away_team"])

stats_teams = set(team_performance["team"])

assert schedule_teams == stats_teams

print("Team identifiers match across datasets.")
assert len(model_data) == len(game_backbone)

print("Game-level merge preserved row count.")
model_data[profile_features].isna().sum()

Team identifiers match across datasets.
Game-level merge preserved row count.


home_pass_epa_per_dropback_last5    173
away_pass_epa_per_dropback_last5    173
home_rush_epa_per_carry_last5       173
away_rush_epa_per_carry_last5       173
home_sack_rate_last5                173
away_sack_rate_last5                173
home_turnovers_last5                173
away_turnovers_last5                173
home_pass_epa_allowed_last5         173
away_pass_epa_allowed_last5         173
home_rush_epa_allowed_last5         173
away_rush_epa_allowed_last5         173
home_def_sack_rate_last5            173
away_def_sack_rate_last5            173
home_takeaways_last5                173
away_takeaways_last5                173
rest_diff                             0
dtype: int64

## Preparing the Machine Learning Dataset

The cleaned game-level dataset is now ready for modeling.

`X` contains the pregame features available to the model.

`y` contains the binary target, where:

- `1` = home team win
- `0` = home team loss

The target is kept separate from the feature matrix to ensure that the model only receives information that would have been available before the game.

In [87]:
X = model_data_clean[profile_features].copy()

y = model_data_clean["home_win"].copy()

In [88]:
print("X shape: ", X.shape)
print("y shape: ", y.shape)

X shape:  (2711, 17)
y shape:  (2711,)


In [89]:
X.head()

,home_pass_epa_per_dropback_last5,away_pass_epa_per_dropback_last5,home_rush_epa_per_carry_last5,away_rush_epa_per_carry_last5,home_sack_rate_last5,away_sack_rate_last5,home_turnovers_last5,away_turnovers_last5,home_pass_epa_allowed_last5,away_pass_epa_allowed_last5,home_rush_epa_allowed_last5,away_rush_epa_allowed_last5,home_def_sack_rate_last5,away_def_sack_rate_last5,home_takeaways_last5,away_takeaways_last5,rest_diff
16,0.170092,-0.273810,-0.243770,-0.193166,0.057143,0.090909,0.0,1.0,-0.124416,-0.517002,-0.197352,-0.189689,0.096154,0.058824,2.0,2.0,0
17,0.524876,0.469451,-0.070528,-0.010526,0.000000,0.058824,0.0,0.0,0.044660,0.395989,0.103916,0.104161,0.039216,0.073171,3.0,1.0,-3
18,-0.008467,-0.124416,-0.146550,-0.197352,0.060606,0.096154,1.0,2.0,-0.344942,0.170092,-0.042704,-0.243770,0.111111,0.057143,3.0,0.0,0
19,0.006034,0.507832,0.228725,-0.348388,0.052632,0.000000,1.0,1.0,0.582160,0.036596,0.160186,-0.187498,0.000000,0.040000,0.0,1.0,0
20,0.408982,0.288580,0.009353,-0.236356,0.000000,0.046512,0.0,3.0,-0.082221,-0.040509,-0.457011,0.085140,0.044444,0.032258,2.0,2.0,0


In [90]:
y.head()

16    0
17    0
18    1
19    0
20    1
Name: home_win, dtype: int64

In [92]:
train_mask = model_data_clean["season"] <= 2023
val_mask = model_data_clean["season"] == 2024
test_mask = model_data_clean["season"] == 2025

In [93]:
X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_val = X.loc[val_mask]
y_val = y.loc[val_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

In [94]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (2200, 17)
Validation: (256, 17)
Test: (255, 17)


In [95]:
print(
    "Total split rows:",
    len(X_train) + len(X_val) + len(X_test)
)

print(
    "Clean dataset rows:",
    len(model_data_clean)
)

Total split rows: 2711
Clean dataset rows: 2711


## Baseline Performance

Before training a machine learning model, a simple baseline is needed.

The first baseline predicts that the home team wins every game.

Because the target variable is encoded as:

- `1` = home win
- `0` = away win

the mean of the target variable represents the proportion of games won by the home team.

A useful machine learning model should outperform this simple rule on unseen data.

In [96]:
train_home_win_rate = y_train.mean()
val_home_win_rate = y_val.mean()
test_home_win_rate = y_test.mean()

print("Training home-win baseline:", train_home_win_rate)
print("Validation home-win baseline:", val_home_win_rate)
print("Test home-win baseline:", test_home_win_rate)

Training home-win baseline: 0.5522727272727272
Validation home-win baseline: 0.5234375
Test home-win baseline: 0.5372549019607843


In [97]:
print(f"Training baseline: {train_home_win_rate:.2%}")
print(f"Validation baseline: {val_home_win_rate:.2%}")
print(f"Test baseline: {test_home_win_rate:.2%}")

Training baseline: 55.23%
Validation baseline: 52.34%
Test baseline: 53.73%
